In [7]:
import pandas as pd
import os
import torch
import numpy as np
from eval import score_heatmap

# 1. Initial Configurations
csv_path = r'C:\Repositories\picard-anomalydetection\data\BCS-DBT-boxes-validation-v2-PHASE-2-Jan-2024.csv'
df_boxes = pd.read_csv(csv_path)

# Put the path to the 'data' folder where the latest .pt files were saved here
heatmaps_folder = r'C:\Repositories\picard-anomalydetection\heatmaps\Breast-Cancer-Screening-DBT\dropout\03-08-2026_17-42-40\data'

# List all files generated by the model
all_generated_files = os.listdir(heatmaps_folder)
auc_results = []

print(f"Starting evaluation based on the {len(df_boxes)} bounding boxes from the CSV...\n")

# 2. Loop through each row of your Bounding Boxes CSV
for index, row in df_boxes.iterrows():
    patient_id = str(row['PatientID'])
    view = str(row['View']).lower()
    slice_num = int(row['Slice'])
    
    # 3. Radar: Look for the heatmap corresponding to this row
    found_heatmap = None
    for filename in all_generated_files:
        lower_name = filename.lower()
        # Check if the Patient and View (rcc, lmlo) are in the filename
        if patient_id.lower() in lower_name and view in lower_name:
            # Check if the slice matches (supports both '_fatia25_' and '_slice025_')
            if f"fatia{slice_num}_" in lower_name or f"slice{slice_num:03d}" in lower_name:
                found_heatmap = filename
                break # Found the file, stop looking
                
    # 4. Evaluation
    if found_heatmap:
        # Extract the exact coordinates from your CSV
        x = int(row['X'])
        y = int(row['Y'])
        width = int(row['Width'])
        height = int(row['Height'])
        
        # PICARD's eval.py requires the format: (top_y, left_x, height, width)
        ground_truth_box = [(y, x, height, width)]
        
        # Load the heatmap tensor generated by your AI
        full_path = os.path.join(heatmaps_folder, found_heatmap)
        heatmap_tensor = torch.load(full_path)
        
        # Calculate the score (Area Under Curve)
        perfect_mask, auc_score = score_heatmap(
            score_type='pixel_AUC', 
            heatmap=heatmap_tensor, 
            bboxes=ground_truth_box
        )
        
        lesion_type = row['Class'] # cancer or benign
        print(f"✅ {patient_id} | {view} | Slice {slice_num} ({lesion_type}) -> AUROC: {auc_score:.4f}")
        
        auc_results.append(auc_score)

# 5. Final Result
print("-" * 50)
if auc_results:
    mean_auc = np.mean(auc_results)
    print(f"🚀 MODEL OVERALL MEAN (AUROC): {mean_auc:.4f}")
    print(f"Total evaluated lesions: {len(auc_results)}")
else:
    print("No matching heatmap was found. Check if the heatmaps_folder path is correct and if the model has already generated the test files.")

Starting evaluation based on the 75 bounding boxes from the CSV...

✅ DBT-P00114 | rmlo | Slice 25 (cancer) -> AUROC: 0.0958
--------------------------------------------------
🚀 MODEL OVERALL MEAN (AUROC): 0.0958
Total evaluated lesions: 1
